In [1]:
import os
import sys

project_root = os.path.abspath('..')
sys.path.append(project_root)

# Somehow this is needed for the imports to work
# Else it takes infinite time to import the modules
from gevent import monkey
monkey.patch_all()

# Now import the required modules
from mu_bench.metrics_exporter.prometheus_exporter import PrometheusExporter
from mu_bench.request_generator.edge_request_generator import EdgeRequestGenerator
from mu_bench.experiment_manager import EdgeExperimentManager

In [2]:
config = {
    "base_host": "172.23.0.5",
    "apps": [
        {
            "name": "app-edge",  # Different name creates different hash
            "port": 31115,
            "traffic_weight": 1,
            "wait_time": {"min": 0.7, "max": 1.5}
        },
        # {
        #     "name": "app-edge-2",  # Different name = different behavior
        #     "port": 31116,
        #     "traffic_weight": 0.9,
        #     "wait_time": {"min": 0.8, "max": 2.5}
        # },
    ],
    "edge_sites": {
        "edge-site-1": {
            "pattern": "urban_peak",
            "description": "Urban site with business peaks",
            "traffic_weight": 1.0
        },
        "edge-site-2": {
            "pattern": "suburban_steady", 
            "description": "Suburban site with steady traffic",
            "traffic_weight": 0.8
        }
    }
}

In [ ]:
prometheus_logger = PrometheusExporter(
    prometheus_url="http://localhost:9090",
    target_namespaces=["app-edge-2"],
    collection_interval_minutes=5,
    metrics_step="30s",
    output_dir="metrics",
    verbose=True,
    create_output_dir=False
)

# Create traffic generator
traffic_generator = EdgeRequestGenerator()

# Create experiment manager
experiment_manager = EdgeExperimentManager(
    prometheus_exporter=prometheus_logger,
    traffic_generator=traffic_generator,
    experiment_name=None,
    output_base_dir="../DATA"
)

# Run experiment

try:
    success = experiment_manager.run_experiment(
        edge_config=config,
        duration_hours=1,        # Real time: 7 hours
        simulated_hours=24,      # Simulated time: 24 hours
        max_users=40,
        spawn_rate=3
    )
except Exception as e:
    print("\n🚫 Experiment interrupted by user")
finally:
    if success:
        print(f"\n🎉 Experiment completed! Results: {experiment_manager.output_directory}")
    else:
        print("\n💥 Experiment failed!")